In [1]:
#from pylibCZIrw import czi as pyczi
#import ipywidgets as widgets
# import json
# from matplotlib import pyplot as plt
# import matplotlib.cm as cm
# import numpy as np
# import os, sys
# from tqdm import tqdm
# from tqdm.contrib import itertools as it
# from matplotlib.patches import Rectangle
# from aicsimageio import AICSImage
# from aicsimageprocessing import resize
# from aicsimageio.writers import OmeTiffWriter
# from aicsimageio.types import PhysicalPixelSizes
import processing_tools as pt
import os

# show the used python env
#print("Using:", sys.executable)
# get current directory
path = os.getcwd()
print("Current Directory", path)
 
# prints parent directory
print(os.path.abspath(os.path.join(path, os.pardir)))



Current Directory e:\code\Mikala\fly_ovaries_processing_pipeline\src
e:\code\Mikala\fly_ovaries_processing_pipeline


In [2]:

# Folder containing the input data
INPUT_FOLDER = r'E:\Projects\Mikala\images\raw\test' #point to folder with the images or folder containing all the subfolders to batch process
PREPROCESSED_FOLDER = r'E:\Projects\Mikala\images\preproc' #point to main folder with all the folders containing preprocessed data
OUTPUT_FOLDER = r'E:\Projects\Mikala\output\test' #point to main folder with all the output

doPreprocessing = True # True for preprocessing / False to skip preprocessing 
targetPxSize = [0.25, 0.14, 0.14] #ZYX
channelMapping = '4321' # 1:VASA, 2:membrane, 3:TJ, 4:DAPI

#Old models
#VASA_model_path = 'D:\\Mikala\\processing_pipeline\\fly_ovaries_processing_pipeline\\models'
#VASA_model_name = 'VASA_xy_pipeline_bright_Mikala'
#TJ_model_path =  "D:\\Mikala\\processing_pipeline\\fly_ovaries_processing_pipeline\\models"
#TJ_model_name = 'TJ_xy_pipeline_norm_Mikala'

# New models as of 10222024
VASA_model_path = r'E:\Projects\Mikala\CP_models'
VASA_model_name = 'VASA_082025set_cp4_xy_yz_xz_08072025_300epoch'
TJ_model_path =  r'E:\Projects\Mikala\CP_models'
TJ_model_name = 'TJ_xy_10222024'



  

## Process single VASA folder

In [3]:
fileList = pt.readFolder(INPUT_FOLDER)

print('Preparing output directory:' )
preprocDatasetFolder = pt.buildOutputFolder(PREPROCESSED_FOLDER, fileList[0].split('_')[0]+'_test')

fullPathFileList = [INPUT_FOLDER  + '\\' + f for f in fileList ]

print('Preprocessing started' )
if doPreprocessing:
    pt.processFiles(fullPathFileList, preprocDatasetFolder, targetPxSize, channelMapping)


print('Initializing cellpose:')
pt.initializeCellPose()

print('Running cellpose segmentation')
VASA_outputFolder = pt.processVASA_cp4(preprocDatasetFolder, OUTPUT_FOLDER, VASA_model_path, VASA_model_name)


print('Processing finished')

VASAsegmentedStats, VASAsegmentedRegions = pt.refineVASASegmentationMasks(VASA_outputFolder, 'VASA')
pt.exportVASASegmentationDetailedStats(VASAsegmentedStats, OUTPUT_FOLDER,fileList[0].split('_')[0])
pt.exportVASASegmentationResults(VASAsegmentedRegions, OUTPUT_FOLDER, fileList[0].split('_')[0])

#VASAstats = pt.processSegmentationMasks(VASA_outputFolder, 'VASA')

#pt.exportVASASegmentationResults(VASAstats, OUTPUT_FOLDER, fileList[0].split('_')[0])

Preprocessing files from directory: E:\Projects\Mikala\images\raw\test
2 files found:
	08212024_1A_processed_ovary1_63x_dapi_488_568_647.czi
	08212024_1A_processed_ovary2_63x_dapi_488_568_647.czi
Preparing output directory:
	[buildOutputFolder] Creating:E:\Projects\Mikala\images\preproc
Preprocessing started
[processFiles] processing file 1 of 2
	Processing channel 4(VASA) as channel 1
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as E:\Projects\Mikala\images\preproc\08212024_test\ch1\08212024_1A_processed_ovary1_ch1_downs.tiff
	Processing channel 3(membrane) as channel 2
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as E:\Projects\Mikala\images\preproc\08212024_test\ch2\08212024_1A_processed_ovary1_ch2_downs.tiff
	Processing channel 2(TJ) as channel 3
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as E:\Projects\Mikala\images\preproc\08212024_test\ch3\08212024_1A_processed_ovary1_ch3_down

c:\Users\amarcosv\AppData\Local\miniforge3\envs\cellpose4_mikala_bioio\Lib\site-packages\cellpose\dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:823.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


Saving output files to directory:E:\Projects\Mikala\output\test\VASA_ch1\08212024
Processing file 2 of 2
Performing prediction on: E:\Projects\Mikala\output\test\VASA_ch1\08212024\08212024_1A_processed_ovary2_ch1_downs.tiff
Saving output files to directory:E:\Projects\Mikala\output\test\VASA_ch1\08212024
Processing finished
[processVASASegmentationMasks] Found 3 outliers in file:  08212024_1A_processed_ovary1_ch1_downs_cp_masks.tif


e:\code\Mikala\fly_ovaries_processing_pipeline\src\processing_tools.py:568: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imsave`.
  io.imsave(os.path.join(masksFolder, maskFile.split('.tif')[0] + '_refined.tif'), refined_labeled_image.astype('uint16'), plugin='tifffile', compression='lzw')
e:\code\Mikala\fly_ovaries_processing_pipeline\src\processing_tools.py:568: FutureWarning: The plugin infrastructure in `skimage.io` is deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not pass additional keyword arguments for plugins (`**plugin_args`). Instead, use `imageio` or other I/O packages directly. See also `skimage.io.imsave`.
  io.imsave(os.path.join(masksFolder, maskFile.split('.tif')[0] + '_refin

[processVASASegmentationMasks] Found 6 outliers in file:  08212024_1A_processed_ovary2_ch1_downs_cp_masks.tif


e:\code\Mikala\fly_ovaries_processing_pipeline\src\processing_tools.py:568: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imsave`.
  io.imsave(os.path.join(masksFolder, maskFile.split('.tif')[0] + '_refined.tif'), refined_labeled_image.astype('uint16'), plugin='tifffile', compression='lzw')
e:\code\Mikala\fly_ovaries_processing_pipeline\src\processing_tools.py:568: FutureWarning: The plugin infrastructure in `skimage.io` is deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not pass additional keyword arguments for plugins (`**plugin_args`). Instead, use `imageio` or other I/O packages directly. See also `skimage.io.imsave`.
  io.imsave(os.path.join(masksFolder, maskFile.split('.tif')[0] + '_refin

## Process singe folder

In [ ]:
fileList = pt.readFolder(INPUT_FOLDER)

print('Preparing output directory:' )
preprocDatasetFolder = pt.buildOutputFolder(PREPROCESSED_FOLDER, fileList[0].split('_')[0]+'_test')

fullPathFileList = [INPUT_FOLDER  + '\\' + f for f in fileList ]

print('Preprocessing started' )
if doPreprocessing:
    pt.processFiles(fullPathFileList, preprocDatasetFolder, targetPxSize, channelMapping)


print('Initializing cellpose:')
pt.initializeCellPose()

print('Running cellpose segmentation')
TJ_outputFolder = pt.processTJ(preprocDatasetFolder, OUTPUT_FOLDER, TJ_model_path, TJ_model_name)
VASA_outputFolder = pt.processVASA(preprocDatasetFolder, OUTPUT_FOLDER, VASA_model_path, VASA_model_name)


print('Processing finished')



#VASAstats = pt.processSegmentationMasks(VASA_outputFolder, 'VASA')
#pt.exportVASASegmentationResults(VASAstats, OUTPUT_FOLDER,fileList[0].split('_')[0])

VASAsegmentedStats, VASAsegmentedRegions = pt.refineVASASegmentationMasks(VASA_outputFolder, 'VASA')
pt.exportVASASegmentationDetailedStats(VASAsegmentedStats, OUTPUT_FOLDER,fileList[0].split('_')[0])
pt.exportVASASegmentationResults(VASAsegmentedRegions, OUTPUT_FOLDER, fileList[0].split('_')[0])


08-Aug-25 16:14:03 - cellpose.models - WARNING  - diam_mean argument are not used in v4.0.1+. Ignoring this argument...


Preprocessing files from directory: D:\Mikala\images\raw\07302025\
10 files found:
	07302025_control29C_dapi_488_555_647_ovary10_63x-Airyscan Processing-09.czi
	07302025_control29C_dapi_488_555_647_ovary11_63x-Airyscan Processing-10.czi
	07302025_control29C_dapi_488_555_647_ovary1_63x-Airyscan Processing-01.czi
	07302025_control29C_dapi_488_555_647_ovary2_63x-Airyscan Processing-02.czi
	07302025_control29C_dapi_488_555_647_ovary3_63x-Airyscan Processing-03.czi
	07302025_control29C_dapi_488_555_647_ovary4_63x-Airyscan Processing-04.czi
	07302025_control29C_dapi_488_555_647_ovary5_63x-Airyscan Processing-05.czi
	07302025_control29C_dapi_488_555_647_ovary6_63x-Airyscan Processing-06.czi
	07302025_control29C_dapi_488_555_647_ovary7_63x-Airyscan Processing-07.czi
	07302025_control29C_dapi_488_555_647_ovary8_63x-Airyscan Processing-08.czi
Preparing output directory:
	[buildOutputFolder] Creating:D:\Mikala\images\preproc
Preprocessing started
Initializing cellpose:
>>> GPU activated? YES
Runn

AttributeError: 'CellposeModel' object has no attribute 'diam_labels'

## Batch process  multiple folders

In [ ]:
INPUT_FOLDER_LIST = os.listdir(INPUT_FOLDER)

for idx, FOLDER in enumerate(INPUT_FOLDER_LIST):
    TARGET_FOLDER = os.path.join(INPUT_FOLDER,FOLDER)
    print('[MAIN] processing folder ' + str(idx+1) + ' of ' +str(len(INPUT_FOLDER_LIST)))
    if not os.path.isdir(TARGET_FOLDER):
        continue

    fileList = pt.readFolder(TARGET_FOLDER)

    print('Preparing output directory:' )
    preprocDatasetFolder = pt.buildOutputFolder(PREPROCESSED_FOLDER, fileList[0].split('_')[0]+'_test')

    fullPathFileList = [TARGET_FOLDER  + '\\' + f for f in fileList ]

    print('Preprocessing started' )
    pt.processFiles(fullPathFileList, preprocDatasetFolder, targetPxSize, channelMapping)


    print('Initializing cellpose:')
    pt.initializeCellPose()

    print('Running cellpose segmentation')
    VASA_outputFolder = pt.processVASA(preprocDatasetFolder, OUTPUT_FOLDER, VASA_model_path, VASA_model_name)
    TJ_outputFolder = pt.processTJ(preprocDatasetFolder, OUTPUT_FOLDER, TJ_model_path, TJ_model_name)


    print('Processing finished')


    VASAstats = pt.processSegmentationMasks(VASA_outputFolder, 'VASA')
    TJstats = pt.processSegmentationMasks(TJ_outputFolder, 'TJ')

    exportdata = pt.mergeSegmentationResults(TJstats, VASAstats)
    pt.exportSegmentationResults(exportdata, OUTPUT_FOLDER, fileList[0].split('_')[0])


